# 第10章　検出（detection）― 病変の位置を四角で示す**『医療診断支援AIを自分で作る（基礎編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-basic

## NMSを数字で追う ― 重複した枠が消える瞬間

In [ ]:
from torchvision.ops import box_iou          # 枠どうしのIoUをまとめて計算してくれるdef nms(boxes, scores, iou_thr=0.5):    order = scores.argsort(descending=True)   # 確信度の高い順に並べる    keep = []    while len(order):        i = order[0]; keep.append(i.item())    # 先頭（最強）を採用        if len(order) == 1: break        ious = box_iou(boxes[i:i+1], boxes[order[1:]])[0]  # 残りとのIoU        order = order[1:][ious <= iou_thr]     # 重なりが小さいものだけ残す    return keep

## 10.11　コードで動かす ― 最小の検出

In [ ]:
import torchfrom torchvision.models.detection import fasterrcnn_resnet50_fpnfrom torchvision.io import read_image, ImageReadModefrom torchvision.transforms.functional import convert_image_dtype# 事前学習済みモデルを読み込み、推論モードにするmodel = fasterrcnn_resnet50_fpn(weights="DEFAULT")model.eval()img = read_image("xray.png", mode=ImageReadMode.RGB)   # 1chのX線でも3chで読む（1chのままでもエラーは出ないが、内部の正規化で暗黙に3ch扱いされ、事前学習の前提と静かにずれる）x = convert_image_dtype(img, torch.float)          # 0〜1 に正規化with torch.no_grad():                              # 勾配計算は不要（推論のみ）    pred = model([x])[0]                            # 1枚を渡し、結果を取り出す# スコア閾値でふるいにかけるkeep = pred["scores"] > 0.5boxes  = pred["boxes"][keep]                       # 残った枠 (N, 4)labels = pred["labels"][keep]                      # クラスIDscores = pred["scores"][keep]                      # 確信度print(f"{len(boxes)} 個の物体を検出（クラスはCOCOのもの）")

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolo11n.pt")                 # 事前学習済みの軽量モデルから開始（転移学習）model.train(data="fracture.yaml",          # 画像とアノテーションの場所を書いた設定ファイル            epochs=100, imgsz=640)          # 100エポック、入力640pxresults = model("new_xray.png", conf=0.10) # 推論。conf の既定は0.25。検証で選んだ閾値を明示する